In [1]:
"""
Dân Trí News Crawler - Crawl articles with pagination
Crawls articles from Dân Trí website: https://dantri.com.vn/the-gioi/trang-1.htm
"""

import requests
from bs4 import BeautifulSoup
import time
import os
import re
from datetime import datetime
from typing import List, Dict, Tuple
from urllib.parse import urljoin
import json

# Configuration
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

OUTPUT_DIR = 'raw_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_SENTENCES = 1000000  # Maximum number of sentences to collect

print("Setup completed successfully!")
print(f"Output directory: {OUTPUT_DIR}")


Setup completed successfully!
Output directory: raw_data


In [2]:
# ==================== Text Processing ====================
def clean_text(text: str) -> str:
    """Clean and normalize Vietnamese text"""
    if not text:
        return ""
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove special characters but keep Vietnamese diacritics
    text = re.sub(r'[\n\r\t]', ' ', text)
    
    return text

def split_into_sentences(text: str) -> List[str]:
    """Split Vietnamese text into sentences"""
    if not text:
        return []
    
    # Vietnamese sentence endings
    text = text.replace('.\n', '.')
    text = text.replace('!\n', '!')
    text = text.replace('?\n', '?')
    
    # Split by sentence endings: . ! ? and also by em dash (–)
    sentences = re.split(r'(?<=[.!?–\-])\s+(?=[A-ZÀỂẦẬĂẰẲẴẶẢẠẪẨễEÊẾỀỂ0-9])', text)
    
    # Clean each sentence and filter out short ones
    cleaned_sentences = []
    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) > 5:  # Keep sentences longer than 5 characters
            cleaned_sentences.append(sentence)
    
    return cleaned_sentences

print("Text processing functions created!")


Text processing functions created!


In [3]:
# ==================== Dân Trí Crawler ====================
def crawl_dan_tri_article(url: str) -> List[str]:
    """
    Crawl content from a single Dân Trí article
    """
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        response.encoding = 'utf-8'
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Remove script and style elements
        for script in soup(['script', 'style']):
            script.decompose()
        
        # Try different selectors for article content
        article_content = None
        
        # Try main selector (most reliable for Dân Trí)
        main_elem = soup.find('main')
        if main_elem:
            article_content = main_elem
        
        # If not found, try article-content
        if not article_content:
            article_content = soup.find('article', class_='article-content')
        
        # Try singular-content
        if not article_content:
            article_content = soup.find('div', class_='singular-content')
        
        # Try article-wrap
        if not article_content:
            article_content = soup.find('div', class_='article-wrap')
        
        if article_content:
            # Get all text from paragraphs
            paragraphs = article_content.find_all('p')
            
            if paragraphs:
                text = '\n'.join([p.get_text() for p in paragraphs])
            else:
                # If no paragraphs, get all text
                text = article_content.get_text()
        else:
            # Fallback: get all text from body
            text = soup.get_text()
        
        # Clean text
        text = clean_text(text)
        
        # Remove common header/footer elements
        text = re.sub(r'(Liên hệ|Chịu trách nhiệm|Tòa soạn|Bản quyền|@Dân trí).*?(?=\n|\Z)', '', text, flags=re.IGNORECASE)
        
        # Split into sentences
        sentences = split_into_sentences(text)
        
        return sentences
    
    except Exception as e:
        print(f"Error crawling {url}: {e}")
        return []

def get_dan_tri_article_urls(page: int, base_url: str = None, category: str = None) -> List[str]:
    """
    Get article URLs from Dân Trí listing page with pagination
    
    Args:
        page: Page number
        base_url: Base URL (e.g., 'https://dantri.com.vn/the-gioi/')
        category: Category name (e.g., 'the-gioi', 'thoi-su', 'kinh-te')
    """
    try:
        # Determine URL
        if base_url:
            # Use provided base URL directly
            if not base_url.endswith('/'):
                base_url += '/'
            url = f"{base_url}trang-{page}.htm"
        elif category:
            # Build URL from category
            url = f"https://dantri.com.vn/{category}/trang-{page}.htm"
        else:
            # Default: the-gioi
            url = f"https://dantri.com.vn/the-gioi/trang-{page}.htm"
        
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        response.encoding = 'utf-8'
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        article_urls = []
        
        # Find all article items (this is the correct structure for Dân Trí)
        article_items = soup.find_all('article', class_='article-item')
        
        for item in article_items:
            # Get the link from article-thumb or content
            link = item.find('a', href=True)
            if link:
                href = link.get('href')
                # Convert relative URLs to absolute
                if not href.startswith('http'):
                    href = urljoin('https://dantri.com.vn', href)
                
                # Filter to actual article links (not events or special pages)
                if 'dantri.com.vn' in href and '/event/' not in href:
                    article_urls.append(href)
        
        # Remove duplicates while preserving order
        article_urls = list(dict.fromkeys(article_urls))
        
        print(f"Found {len(article_urls)} articles on page {page}")
        return article_urls
    
    except Exception as e:
        print(f"Error getting article URLs from page {page}: {e}")
        return []

print("Dân Trí crawler functions updated!")

Dân Trí crawler functions updated!


In [4]:
# ==================== Main Crawling Function ====================
def crawl_dan_tri_pagination(start_page: int = 1, end_page: int = 10, delay: float = 1.0,
                              category: str = None, base_url: str = None):
    """
    Crawl Dân Trí articles with pagination
    
    Args:
        start_page: Starting page number (default: 1)
        end_page: Ending page number (default: 10)
        delay: Delay between requests in seconds (default: 1.0)
        category: Category name (e.g., 'the-gioi', 'thoi-su', 'kinh-te')
        base_url: Base URL (e.g., 'https://dantri.com.vn/the-gioi/')
    """
    
    all_sentences = []
    all_sources = []
    
    output_file = os.path.join(OUTPUT_DIR, 'crawled_news_pagination.txt')
    log_file = os.path.join(OUTPUT_DIR, 'crawl_log_dan_tri.txt')
    
    # Append mode
    mode = 'a'
    
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Determine URL display name
    if base_url:
        url_display = base_url
    elif category:
        url_display = f"https://dantri.com.vn/{category}/"
    else:
        url_display = "https://dantri.com.vn/the-gioi/"
    
    with open(log_file, mode, encoding='utf-8') as log:
        log.write(f"\n{'='*60}\n")
        log.write(f"Crawl started at {timestamp}\n")
        log.write(f"URL: {url_display}\n")
        log.write(f"Pages: {start_page} to {end_page}\n")
        log.write(f"{'='*60}\n\n")
    
    total_articles = 0
    total_sentences = 0
    
    try:
        for page in range(start_page, end_page + 1):
            print(f"\n{'='*60}")
            print(f"Crawling page {page}...")
            print(f"{'='*60}")
            
            # Get article URLs from current page
            article_urls = get_dan_tri_article_urls(page, base_url=base_url, category=category)
            
            if not article_urls:
                print(f"No articles found on page {page}, stopping...")
                break
            
            page_sentences_count = 0
            
            for idx, article_url in enumerate(article_urls, 1):
                print(f"  [{idx}/{len(article_urls)}] Crawling: {article_url[:80]}...")
                
                # Crawl article content
                sentences = crawl_dan_tri_article(article_url)
                
                if sentences:
                    all_sentences.extend(sentences)
                    all_sources.extend([article_url] * len(sentences))
                    page_sentences_count += len(sentences)
                    total_articles += 1
                    total_sentences += len(sentences)
                    
                    # Write sentences to file in real-time
                    with open(output_file, 'a', encoding='utf-8') as f:
                        for sentence in sentences:
                            f.write(sentence + '\n')
                
                # Delay between requests
                time.sleep(delay)
                
                # Check if we've reached max sentences
                if total_sentences >= MAX_SENTENCES:
                    print(f"\nReached maximum sentences limit ({MAX_SENTENCES})")
                    break
            
            print(f"Page {page} completed: {page_sentences_count} sentences from {len(article_urls)} articles")
            
            # Log progress
            with open(log_file, 'a', encoding='utf-8') as log:
                log.write(f"Page {page}: {len(article_urls)} articles, {page_sentences_count} sentences\n")
            
            # Delay between pages
            time.sleep(delay * 2)
            
            if total_sentences >= MAX_SENTENCES:
                break
    
    except KeyboardInterrupt:
        print("\n\nCrawling interrupted by user")
    except Exception as e:
        print(f"\nError during crawling: {e}")
        with open(log_file, 'a', encoding='utf-8') as log:
            log.write(f"Error: {e}\n")
    
    finally:
        # Final summary
        print(f"\n{'='*60}")
        print(f"Crawling completed!")
        print(f"{'='*60}")
        print(f"Total articles crawled: {total_articles}")
        print(f"Total sentences collected: {total_sentences}")
        print(f"Output file: {output_file}")
        print(f"Log file: {log_file}")
        
        # Write final summary to log
        with open(log_file, 'a', encoding='utf-8') as log:
            log.write(f"\n{'='*60}\n")
            log.write(f"Summary:\n")
            log.write(f"  Total articles: {total_articles}\n")
            log.write(f"  Total sentences: {total_sentences}\n")
            log.write(f"  Crawled at: {timestamp}\n")
            log.write(f"{'='*60}\n")

print("Main crawling function updated!")

Main crawling function updated!


In [ ]:
# 🚀 CRAWL TẤT CẢ CÁC CATEGORIES - Automatic batch crawl (no input needed)

print("=" * 70)
print("🚀 CRAWL TẤT CẢ CÁC THỂ LOẠI TỬ AVAILABLE_CATEGORIES")
print("=" * 70)

# Available categories - automatically crawled
AVAILABLE_CATEGORIES = {
    '1': ('the-gioi', 'Thế giới'),
    '2': ('thoi-su', 'Thời sự'),
    '3': ('phap-luat', 'Pháp luật'),
    '4': ('suc-khoe', 'Sức khỏe'),
    '5': ('doi-song', 'Đời sống'),
    '6': ('du-lich', 'Du lịch'),
    '7': ('kinh-doanh', 'Kinh doanh'),
    '8': ('bat-dong-san', 'Bất động sản'),
    '9': ('the-thao', 'Thể thao'),
    '10': ('giai-tri', 'Giải trí'),
    '11': ('giao-duc', 'Giáo dục'),
    '12': ('o-to-xe-may', 'Ô tô - Xe máy'),
    '13': ('noi-vu', 'Nội vụ'),
    '14': ('tam-long-nhan-ai', 'Tấm lòng nhân ái'),
    '15': ('cong-nghe', 'Công nghệ'),
    '16': ('lao-dong-viec-lam', 'Lao động - Việc làm'),
    '17': ('tinh-yeu-gioi-tinh', 'Tình yêu - Giới tính'),
    '18': ('khoa-hoc', 'Khoa học'),
}

# Extract all categories
all_categories = [cat[0] for cat in AVAILABLE_CATEGORIES.values()]
# Remove duplicates while preserving order
all_categories = list(dict.fromkeys(all_categories))

print(f"\n📊 Tổng số categories: {len(all_categories)}")
for i, cat in enumerate(all_categories, 1):
    print(f"   {i:2d}. {cat}")

# ========== CRAWLING PARAMETERS ==========
START_PAGE_BATCH = 1
END_PAGE_BATCH = 20              # Pages per category
DELAY_BATCH = 1.0               # Delay between requests
REST_BETWEEN_CATEGORIES = 3     # Rest time between categories (seconds)
# ==========================================

print("\n" + "=" * 70)
print(f"⏱️  Bắt đầu crawl {len(all_categories)} categories")
print(f"📄 Pages: {START_PAGE_BATCH}-{END_PAGE_BATCH} mỗi category")
print(f"⏸️  Delay: {DELAY_BATCH}s | Rest: {REST_BETWEEN_CATEGORIES}s")
print("=" * 70)

results = {}

for idx, category in enumerate(all_categories, 1):
    print(f"\n[{idx}/{len(all_categories)}] 📝 Crawling: {category}")
    print("-" * 70)
    
    try:
        crawl_dan_tri_pagination(
            start_page=START_PAGE_BATCH,
            end_page=END_PAGE_BATCH,
            delay=DELAY_BATCH,
            category=category,
            base_url=None
        )
        
        results[category] = "✅"
        
    except Exception as e:
        results[category] = f"❌"
        print(f"❌ {category} failed: {str(e)}")
    
    # Rest between categories (except for last one)
    if idx < len(all_categories):
        print(f"⏳ Waiting {REST_BETWEEN_CATEGORIES}s...")
        time.sleep(REST_BETWEEN_CATEGORIES)

print("\n" + "=" * 70)
print("📊 BATCH CRAWL COMPLETED")
print("=" * 70)

success_count = sum(1 for v in results.values() if "✅" in v)

print(f"\n✅ Success: {success_count}/{len(all_categories)}")

print("\n📋 Kết quả:")
for cat, status in results.items():
    print(f"   {status} {cat}")

print(f"\n{'='*70}")
print(f"✅ Xong! Kiểm tra dữ liệu trong raw_data/")
print(f"{'='*70}")


🚀 CRAWL TẤT CẢ CÁC THỂ LOẠI TỬ AVAILABLE_CATEGORIES

📊 Tổng số categories: 18
    1. the-gioi
    2. thoi-su
    3. phap-luat
    4. suc-khoe
    5. doi-song
    6. du-lich
    7. kinh-doanh
    8. bat-dong-san
    9. the-thao
   10. giai-tri
   11. giao-duc
   12. o-to-xe-may
   13. noi-vu
   14. tam-long-nhan-ai
   15. cong-nghe
   16. lao-dong-viec-lam
   17. tinh-yeu-gioi-tinh
   18. khoa-hoc

⏱️  Bắt đầu crawl 18 categories
📄 Pages: 1-20 mỗi category
⏸️  Delay: 1.0s | Rest: 3s

[1/18] 📝 Crawling: the-gioi
----------------------------------------------------------------------

Crawling page 1...
Found 33 articles on page 1
  [1/33] Crawling: https://dantri.com.vn/the-gioi/iran-xac-nhan-tu-lenh-hai-quan-irgc-thiet-mang-20...
  [2/33] Crawling: https://dantri.com.vn/the-gioi/quoc-hoi-iran-xem-xet-rut-khoi-hiep-uoc-khong-pho...
  [3/33] Crawling: https://dantri.com.vn/the-gioi/tong-thong-trump-tinh-xay-to-hop-quan-su-ngam-ben...
  [4/33] Crawling: https://dantri.com.vn/the-gioi/iran-t